# 00. Master Architecture & Complete System Walkthrough

---

### Academic Information & Course Attribution
* **Institution:** University of San Diego (USD), Shiley-Marcos School of Engineering
* **Degree Program:** Master of Science in Applied Artificial Intelligence (MS-AAI)
* **Course:** **AAI-520-IN2: Natural Language Processing and Generative AI**
* **Instructor:** **Prof. Rajesh Soni, M.S.**
* **Project Title:** **Autonomous Multi-Agent Equity Research & Factual Grounding System**
* **Project Engineering Team:**
  1. **Sai Krishna Akula** (`nakula@sandiego.edu`)
  2. **Ashok Kumar Bhairwal** (`abhairwal@sandiego.edu`)
  3. **Diaesh Antony** (`dantony@sandiego.edu`)

---

## 1. Executive Summary & Problem Framing

### The Financial Challenge
Institutional equity research demands synthesizing massive volumes of heterogeneous, high-velocity financial information:
- **SEC 10-K & 10-Q Filings:** Unstructured legal text, footnotes, balance sheet debt covenants, and management discussion (MD&A).
- **Fundamental & Valuation Metrics:** Real-time earnings reports, revenue trajectories, gross margins, EBITDA, and valuation multiples.
- **Market Technicians & Quantitative Signals:** OHLCV price action, moving averages (50-day / 200-day SMAs), RSI momentum oscillators, and MACD trend shifts.
- **Real-Time News Catalysts & Sentiment:** Breaking news headlines, regulatory rulings, product launches, and supply-chain developments.

A human Wall Street equity analyst takes between **6 to 12 hours** to gather, verify, calculate, and synthesize an institutional briefing for a single equity.

### The Generative AI Dilemma: Why Single LLM Prompts Fail
When financial firms attempt to automate research using a single LLM prompt or a simple sequential chain (LangChain linear chains), four critical failure modes occur:
1. **Parametric Hallucinations:** Generative models predict plausible next tokens from static pre-trained weights. When asked for quarterly revenue, operating income, or debt figures, they frequently generate realistic but completely fabricated numbers.
2. **Context Window Degradation ("Lost-in-the-Middle"):** Dumping 100+ pages of SEC filings and earnings releases into a single prompt degrades attention. The LLM glosses over critical debt covenants or footnoted liabilities.
3. **Absence of Real-Time Grounding:** Pre-trained weights possess a fixed knowledge cutoff date. They cannot access current stock quotes, bond yields, or breaking morning news without external tool integrations.
4. **Lack of Critical Reflection & Verification:** Single prompts have no mechanism to audit their own arithmetic or fact-check their assertions against raw market evidence.

### Our Solution: Autonomous Multi-Agent Blackboard Architecture
To solve these failure modes, we architected an **institutional-grade, multi-agent equity research platform**. Instead of relying on a monolithic prompt, our system decouples research across **specialized autonomous agents** coordinated over a shared **Blackboard Architecture (`StateGraph`)** with an adversarial **Evaluator-Optimizer Factual Grounding Loop**.

## 2. Core Concept 1: What is an Agent?

### Definition: Agent vs. Monolithic LLM
A standard LLM is a stateless function: $f(\text{prompt}) \to \text{response}$. It cannot ask questions, invoke external calculators, retrieve live files, or course-correct when an initial assumption is wrong.

An **Agent** is an autonomous software entity that pairs an LLM (the reasoning brain) with:
1. **A Defined Role / Persona:** Clear system prompt boundaries that constrain its expertise and behavioral directives.
2. **Tools:** Callable functions (APIs, databases, scrapers) that ground its actions in real-world facts.
3. **The ReAct Execution Loop (Reasoning + Acting + Observing):** An iterative loop where the agent:
   - **Thinks:** Analyzes the prompt and decides what step to take next.
   - **Acts:** Issues a structured tool call (e.g., `lookup_ticker(query="NVDA")`).
   - **Observes:** Receives deterministic output from the executed tool.
   - **Reflects:** Synthesizes the observation and decides whether more tools are needed or if it can formulate the final answer.

```
       ┌────────────────────────────────────────────────────────┐
       │                 THE AGENT ReAct LOOP                   │
       │                                                        │
       │  Prompt ──▶ [LLM Reason / Thought]                     │
       │                     │                                  │
       │                     ▼                                  │
       │             Has Tool Call Needed?                      │
       │                 /        \                             │
       │         [YES]  /          \  [NO]                      │
       │               ▼            ▼                           │
       │        [Tool Action]   [Final Answer Formulated]       │
       │               │                                        │
       │               ▼                                        │
       │        [Observation] ──▶ (Append to Context & Repeat)  │
       └────────────────────────────────────────────────────────┘
```

### Main Functions of an Agent in Our Framework
* **`__init__(name, role, tools, llm)`:** Sets up agent persona, registers tool definitions, and initializes message memory.
* **`run(prompt)`:** Executes the multi-turn ReAct loop until the agent produces a final answer without requesting further tools.
* **`_execute_tool_call(call)`:** Safely executes the requested tool, handles schema errors, and captures the trace step.
* **`as_tool()`:** Wraps the agent itself as a callable tool, enabling an Orchestrator/Supervisor agent to delegate complex sub-tasks to it.

In [1]:
import sys
from pathlib import Path

# Resolve project src directory
project_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "src").exists():
        project_root = candidate
        break
sys.path.insert(0, str(project_root / "src"))

from research_agent.agent_framework import (
    END,
    Agent,
    GraphMemory,
    GraphState,
    StateGraph,
    Tool,
)
from research_agent.interfaces import get_llm

# Initialize shared LLM instance (Ollama Cloud gpt-oss:120b)
llm = get_llm()
print("Framework imported successfully. LLM client initialized.")

Framework imported successfully. LLM client initialized.


In [2]:
# Demonstrate Concept 1 & 2: Creating a custom Tool and an Agent
def calculate_financial_health_score(operating_margin: float, debt_to_equity: float) -> dict:
    """Calculates a composite financial health score from operating margin and leverage.

    Args:
        operating_margin: Operating margin percentage (e.g. 62.0 for 62%).
        debt_to_equity: Debt to equity ratio (e.g. 0.45).
    """
    score = (operating_margin * 0.7) - (debt_to_equity * 20.0)
    verdict = "Strong" if score > 30 else "Moderate" if score > 10 else "High Risk"
    return {"composite_score": round(score, 2), "health_verdict": verdict}


# Wrap function in Tool descriptor with schema
health_tool = Tool(
    name="calculate_financial_health_score",
    description="Calculate composite financial health score from margin and debt ratio.",
    func=calculate_financial_health_score,
)

# Instantiate a specialized Valuation Agent with the tool
analyst_agent = Agent(
    name="FinancialHealthSpecialist",
    role="Evaluate company balance sheet resilience using the health score tool.",
    llm=llm,
    tools=[health_tool],
    max_iterations=4,
)

# Run the agent with a targeted financial inquiry
query = "Evaluate NVDA financial health if operating margin is 62.0% and debt-to-equity is 0.42."
response = analyst_agent.run(query)

print(f"Agent Name     : {analyst_agent.name}")
print(f"Iterations Run : {response.iterations}")
print(f"Tools Used     : {[t.name for t in analyst_agent.tools.values()]}")
print(f"Trace Steps    : {[step.step_type for step in response.trace]}")
print(f"\n--- AGENT FINAL ANSWER ---\n{response.content}")

Agent Name     : FinancialHealthSpecialist
Iterations Run : 2
Tools Used     : ['calculate_financial_health_score']
Trace Steps    : ['tool_call', 'tool_result', 'thought', 'final_answer']

--- AGENT FINAL ANSWER ---
**Financial Health Evaluation for NVIDIA (NVDA)**  

- **Operating Margin:** 62.0%  
- **Debt‑to‑Equity Ratio:** 0.42  

Using the provided **calculate_financial_health_score** tool, the composite financial health score is **35.0**, leading to a **“Strong”** health verdict. This indicates that, given the high operating margin and modest leverage, NVIDIA is in a solid financial position.


## 3. Core Concept 2: What are Tools? Why are they Essential?

### The Essential Function of Tools
Large Language Models are probabilistic next-token predictors. They have no internal calculator, no direct access to SEC databases, and no real-time price feeds.

A **Tool** is a deterministic Python function exposed to the LLM via a structured schema (JSON function signature). The LLM does not execute code itself; instead, it outputs a structured function call:
```json
{
  "name": "lookup_ticker",
  "arguments": {"query": "NVIDIA"}
}
```
The framework executes the deterministic Python function on the host machine, captures the return value (e.g. `{"ticker": "NVDA", "company": "NVIDIA Corporation"}`), and feeds it back to the LLM as an **Observation**.

### Why Tools Eliminate Hallucinations
1. **Mathematical Accuracy:** Operating margins, debt ratios, and YoY growth percentages are computed in Python code, avoiding LLM arithmetic errors.
2. **Temporal Grounding:** Real-time prices, 52-week ranges, and macroeconomic interest rates are pulled live from Yahoo Finance and Alpha Vantage.
3. **Regulatory Verification:** Risk disclosures and revenue breakdowns are retrieved directly from SEC EDGAR 10-K and 10-Q filings.

### Our Suite of Specialized Financial Tools
| Tool Name | Source Provider | Objective & Output |
| :--- | :--- | :--- |
| `lookup_ticker` | Yahoo Finance | Resolves company names/queries to validated ticker symbols. |
| `get_earnings_data` | Yahoo Finance / SEC | Extracts quarterly revenue, gross margins, EBITDA, and net income. |
| `search_sec_filings` | SEC EDGAR Direct | Retrieves official 10-K and 10-Q filing disclosures and legal risk factors. |
| `get_market_technicals` | Yahoo Finance | Computes 50-day / 200-day SMAs, 14-day RSI oscillator, and 52-week trading ranges. |
| `get_macro_indicators` | Alpha Vantage / Fed | Retrieves Federal Funds rates, 10-year Treasury yields, and macro context. |
| `get_news_data` | Yahoo Finance / News RSS | Pulls recent market catalysts, media headlines, and computes net sentiment. |

## 4. Core Concept 3: What is State? The Blackboard Architecture

### The Problem with Linear Prompt Chaining
In traditional linear chains (`Agent A -> Agent B -> Agent C`), prompt strings are passed consecutively. As the chain grows:
* Text accumulates exponentially, causing context window exhaustion.
* Critical quantitative facts generated by Agent A get diluted or lost when Agent B summarizes them for Agent C.
* It is impossible for Agent C to write back corrections to Agent A.

### The Solution: Blackboard Architecture (`GraphState`)
In our framework, all agents collaborate through a centralized **Blackboard Architecture** implemented via `GraphState`.

`GraphState` is a shared, typed data structure representing the single source of truth across the entire research lifecycle:
* `question`: The user's input query or research objective.
* `active_ticker`: The resolved stock symbol (e.g., `"NVDA"`).
* `agents`: A dictionary storing the structured outputs produced by each specialist agent (`state.agents["EarningsAnalyst"]`, `state.agents["MarketAnalyst"]`, etc.).
* `tools`: A dictionary preserving raw tool observations (`state.tools["get_earnings_data"]`, `state.tools["search_sec_filings"]`). This allows downstream auditor agents to verify claims against raw data.
* `trajectory`: An immutable audit log tracking every node execution, timestamp, tool invocation, and state delta.
* `output`: The final synthesized institutional equity briefing released to the user.

```
       ┌────────────────────────────────────────────────────────┐
       │               SHARED BLACKBOARD (GraphState)           │
       │                                                        │
       │  • active_ticker : "NVDA"                              │
       │  • agents        : {EarningsAnalyst: ..., News: ...}   │
       │  • tools         : {get_earnings_data: {rev: 30.04B}}  │
       │  • review_status : "approved"                          │
       │  • trajectory    : [Step 1, Step 2, Step 3, ...]       │
       │  • output        : "Final Institutional Brief..."      │
       └───────────────────────────┬────────────────────────────┘
               ▲                   ▲                   ▲
               │                   │                   │
               ▼                   ▼                   ▼
        EarningsAnalyst     MarketAnalyst         Critic/Refiner
        (reads/writes)      (reads/writes)        (audits blackboard)
```

In [3]:
# Demonstrate Concept 3 & 4: State updates and GraphMemory persistence
memory = GraphMemory()

# Create initial state on the blackboard
state = GraphState(question="Analyze NVIDIA Q2 fundamentals and margins")
state.active_ticker = "NVDA"

# Specialist 1 writes fundamental findings to the blackboard
state.agents["EarningsAnalyst"] = "Q2 Revenue: $30.04B (+122% YoY), Gross Margin: 75.1%."
state.tools["get_earnings_data"] = {"revenue_b": 30.04, "gross_margin_pct": 75.1, "ebitda_b": 18.64}

# Specialist 2 writes market technician findings to the blackboard
state.agents["MarketAnalyst"] = "Current Price: $128.50, 50-day SMA: $121.30, 14-day RSI: 63.34."
state.tools["get_market_technicals"] = {"price": 128.50, "sma_50": 121.30, "rsi_14": 63.34}

# Checkpoint state to transient GraphMemory
checkpoint_id = memory.write(state)

print(f"State Checkpointed ID : {checkpoint_id}")
print(f"Active Ticker         : {state.active_ticker}")
print(f"Agents on Blackboard  : {list(state.agents.keys())}")
print(f"Raw Tools Stored      : {list(state.tools.keys())}")
print(f"Memory State IDs      : {memory.list_state_ids()}")

State Checkpointed ID : state_20261012_235130_25adad
Active Ticker         : NVDA
Agents on Blackboard  : ['EarningsAnalyst', 'MarketAnalyst']
Raw Tools Stored      : ['get_earnings_data', 'get_market_technicals']
Memory State IDs      : [
  "state_20261012_235130_25adad"
]


## 5. Core Concept 4: What is Memory?

### Memory Bifurcation: Two Operational Horizons
A production AI system requires distinct memory horizons: mixing short-term workflow execution state with long-term conversational memory leads to context pollution and cache staleness. We bifurcate memory into two layers:

```
┌────────────────────────────────────────────────────────────────────────┐
│                          MEMORY ARCHITECTURE                           │
├───────────────────────────────────┬────────────────────────────────────┤
│ 1. Transient Memory (GraphMemory) │ 2. Persistent Memory (SessionStore)│
├───────────────────────────────────┼────────────────────────────────────┤
│ • Horizon: Single workflow run    │ • Horizon: Multi-turn chat session │
│ • Storage: In-memory / JSON diffs │ • Storage: SQLite Relational DB    │
│ • Contents: Intermediate traces,  │ • Contents: Message turns, rolling │
│   blackboard updates, node deltas │   summaries, TTL cached briefs     │
│ • Purpose: Agent coordination &   │ • Purpose: Multi-turn conversational│
│   execution telemetry auditing    │   continuity & entity recall       │
└───────────────────────────────────┴────────────────────────────────────┘
```

### Why Persistent SessionStore is Critical for Equities
When an investor asks:
1. *"Analyse NVDA."* (System executes a 30-second multi-agent research workflow and caches the brief).
2. *"What are its main supply chain risks?"*
The system should NOT re-run all 7 agents or re-query Yahoo Finance. The `SessionStore` recognizes that `active_ticker == "NVDA"`, retrieves the cached brief, and answers the follow-up instantly from conversational context.

## 6. Core Concept 5: What is StateGraph?

### Definition & Architecture
`StateGraph` is a graph-based workflow orchestration engine inspired by LangGraph and classical finite-state machines:
1. **Nodes:** Units of computation. A node can be:
   - An autonomous `Agent` (e.g. `ResearchPlanner`, `DraftWriter`).
   - A list of parallel agents executed concurrently.
   - A deterministic Python function (e.g. `critic_node`, `release_node`).
2. **Edges:** Directed transitions connecting node $A \to node B$.
3. **Conditional Edges:** Dynamic routers that evaluate runtime state and select the next node:
   - If user asks a general conceptual question (`"What is EBITDA?"`), the planner routes directly to `END`, bypassing specialist agents.
   - If user asks for company research (`"Analyse NVDA"`), the planner routes to the `orchestrator`.
4. **Cycles (Feedback Loops):** Unlike linear DAGs, `StateGraph` supports cyclical edges. When the `critic_node` detects factual discrepancies or missing metrics, it routes back to `refiner_node`, which refines the draft and loops back to the critic until the review score reaches $\ge 8.0/10$.

In [4]:
# Demonstrate Concept 5: Assembling and compiling a StateGraph
class SimpleState(GraphState):
    counter: int = 0
    message: str = ""


def increment_node(state: SimpleState) -> dict:
    return {"counter": state.counter + 1, "message": f"Counter reached {state.counter + 1}"}


def audit_gate(state: SimpleState) -> str:
    # Router function: decide whether to loop or finish
    return "continue" if state.counter < 3 else "done"


# Build graph topology
mini_graph = StateGraph(state_schema=SimpleState)
mini_graph.add_node("incrementer", increment_node)
mini_graph.set_entry_point("incrementer")
mini_graph.add_conditional_edges(
    "incrementer",
    audit_gate,
    {"continue": "incrementer", "done": END},
)

compiled_mini = mini_graph.compile()
result = compiled_mini.invoke({"counter": 0})

print("Mini StateGraph compilation and cycle execution successful:")
print(f"Final Counter Value : {result.counter}")
print(f"Final Message       : {result.message}")
print(f"Total Trajectory Steps: {len(result.trajectory)}")

Mini StateGraph compilation and cycle execution successful:
Final Counter Value : 3
Final Message       : Counter reached 3
Total Trajectory Steps: 3


## 7. The Institutional Multi-Agent Workflow

Below is the complete architectural topology of our 7-agent research platform:

```
[Start] ──▶ planner (ResearchPlanner)
                 │
                 ├──[General Concept / Greeting]──▶ [End] (Instant Answer)
                 │
                 └──[RESEARCH_REQUIRED: Validated Ticker]
                            │
                            ▼
                    orchestrator (ResearchRouter)
                            │
          ┌─────────────────┼─────────────────┐
          ▼                 ▼                 ▼
   EarningsAnalyst   MarketAnalyst     NewsAnalyst
   (SEC & Income)   (RSI & OHLCV)   (Catalysts)
          └─────────────────┬─────────────────┘
                            ▼
                    draft_writer (DraftWriter)
                            │
                            ▼
                      critic_node
                       /       \
      [score < 8.0]   /         \   [score >= 8.0]
                     ▼           ▼
               refiner_node   release_node ──▶ [End]
                     │
                     └─▶ (Loop back to critic)
```

### The Specialist Agent Team
| Stage | Node Name | Agent / Function | Core Objective | Primary Tools Used |
| :---: | :--- | :--- | :--- | :--- |
| **1** | `planner` | `ResearchPlanner` | Classifies intent, resolves ticker via Yahoo Finance, or answers concepts. | `lookup_ticker`, `memory.as_tools()` |
| **2** | `orchestrator`| `ResearchRouter` | Decomposes goals and delegates sub-tasks to specialist analysts via ReAct. | `EarningsAnalyst`, `MarketAnalyst`, `NewsAnalyst` |
| **3** | Specialist 1 | `EarningsAnalyst` | Analyzes quarterly revenue, gross margins, EBITDA, and SEC 10-K/10-Q disclosures. | `get_earnings_data`, `search_sec_filings` |
| **4** | Specialist 2 | `MarketAnalyst` | Analyzes price action, 50/200 SMAs, 14-day RSI, and macroeconomic interest rates. | `get_market_technicals`, `get_macro_indicators` |
| **5** | Specialist 3 | `NewsAnalyst` | Gathers breaking news headlines, market catalysts, and computes net sentiment. | `get_news_data` |
| **6** | `draft_writer`| `DraftWriter` | Synthesizes blackboard evidence into a structured 4-pillar institutional report. | Shared Blackboard (`GraphState`) |
| **7** | `critic` / `refiner`| Review Loop | **Factual Grounding Gate:** Cross-checks numbers against raw tool observations. | `critic_node`, `refiner_node` |
| **8** | `release` | `release_node` | Strips audit metadata and publishes the verified institutional briefing. | `release_node` |

In [5]:
from research_agent.workflows.graph import build_research_workflow

# Compile the full institutional multi-agent workflow
full_memory = GraphMemory()
workflow = build_research_workflow(llm=llm, memory=full_memory)

print("Institutional Multi-Agent Workflow compiled successfully.")
print(f"Registered Graph Nodes: {list(workflow.graph.nodes.keys())}")
print("\nWorkflow Architecture Topology Diagram:")
workflow.draw(as_image=True)

Institutional Multi-Agent Workflow compiled successfully.
Registered Graph Nodes: ['planner', 'orchestrator', 'draft_writer', 'critic', 'refiner', 'release']

Workflow Architecture Topology Diagram:
<IPython.core.display.Image object>


In [6]:
# Execute live multi-agent research against real market data for NVIDIA (NVDA)
query = (
    "Provide an institutional equity research briefing on NVDA (NVIDIA Corporation)"
    " covering fundamentals, valuation, technical momentum, and news catalysts."
)

print(f"Initiating Autonomous Multi-Agent Research for: '{query}'\n")


def _progress_listener(msg: str) -> None:
    print(f"  {msg}")


state = workflow.invoke({"question": query}, on_progress=_progress_listener)
print("\n[Workflow Execution Completed Successfully]")
ticker_val = getattr(state, "active_ticker", getattr(state, "ticker", "NVDA"))
print(f"Resolved Ticker : {ticker_val}")
print(f"Review Status   : {getattr(state, 'review_status', 'Evaluated')}")
print(f"Trajectory Steps: {len(state.trajectory)}")

Initiating Autonomous Multi-Agent Research for: 'Provide an institutional equity research briefing on NVDA (NVIDIA Corporation) covering fundamentals, valuation, technical momentum, and news catalysts.'

  Thinking: Analyzing query intent and resolving company ticker symbol...
  Tool: Querying Yahoo Finance symbol lookup for 'NVDA'...
  Thinking: Decomposing research brief and dispatching specialist analyst team...
  Agent: [Earningsanalyst] Dispatched specialist research sub-task...
  Tool: Fetching fundamental quarterly revenue, gross margins, and operating income for NVDA...
  Tool: Searching SEC EDGAR 10-K/10-Q filing disclosures for NVDA...
  Tool: Searching SEC EDGAR 10-K/10-Q filing disclosures for NVDA...
  Tool: Searching SEC EDGAR 10-K/10-Q filing disclosures for NVDA...
  Tool: Searching SEC EDGAR 10-K/10-Q filing disclosures for NVDA...
  Agent: [Marketanalyst] Dispatched specialist research sub-task...
  Tool: Calculating 50/200-day SMAs, 14-day RSI, and trading ranges for

In [7]:
print("================================================================================")
print("                       EXECUTION TRAJECTORY & TELEMETRY                        ")
print("================================================================================")

for step in state.trajectory:
    step_no = step["step"]
    node = step["node"]
    executor = step.get("executor", "system")
    summary = step.get("summary", "")
    print(f"\n[Step {step_no}] Node '{node}' [{executor}]")
    print(f"  ↳ Summary: {summary}")

    # Inspect Step 1: Planner tool calls
    if node == "planner" and step.get("activities"):
        for act in step["activities"]:
            args = ", ".join(f"{k}={v!r}" for k, v in act.get("arguments", {}).items())
            print(f"    • Tool Invoked : {act['tool']}({args})")
            obs = act.get("observation")
            if isinstance(obs, dict):
                obs_str = ", ".join(
                    f"{k}: {v}" for k, v in obs.items() if k in ("ticker", "company", "source")
                )
            else:
                obs_str = str(obs)
            print(f"    • Tool Result  : {{{obs_str}}}")

    # Inspect Step 2: Orchestrator delegations
    elif node == "orchestrator":
        subagents = step.get("subagents", [])
        delegations = step.get("delegations", {})
        if not subagents:
            subagents = [
                sa
                for sa in ["EarningsAnalyst", "MarketAnalyst", "NewsAnalyst"]
                if sa in state.agents
            ]
            delegations = {sa: list(state.tools.get(sa, {}).keys()) for sa in subagents}
        if subagents:
            print(f"    • Subagents Dispatched : {', '.join(subagents)}")
            for sa, tools in delegations.items():
                print(f"      - {sa} executed tools: {tools}")

    # Inspect Review Loop
    elif node in ("critic_node", "critic"):
        print(
            f"    • Audit Gate Outcome: {step.get('updates', {}).get('review_status', 'Evaluated')}"
        )

                       EXECUTION TRAJECTORY & TELEMETRY                        

[Step 1] Node 'planner' [ResearchPlanner]
  ↳ Summary: {"research_required": true, "ticker": "NVDA"}
    • Tool Invoked : lookup_ticker(company_name='NVIDIA Corporation', query='NVDA')
    • Tool Result  : {{
  "ticker": "NVDA",
  "company": "NVIDIA Corporation",
  "source": "yahoo_finance"
}}

[Step 2] Node 'orchestrator' [ResearchRouter]
  ↳ Summary: **NVDA – Institutional Equity Research Briefing** *(Compiled 12 Oct 2026 – All f...
    • Subagents Dispatched : EarningsAnalyst, MarketAnalyst, NewsAnalyst
      - EarningsAnalyst executed tools: ['get_earnings_data', 'search_sec_filings']
      - MarketAnalyst executed tools: ['get_market_technicals', 'get_macro_indicators']
      - NewsAnalyst executed tools: ['get_news_data']

[Step 3] Node 'draft_writer' [DraftWriter]
  ↳ Summary: # Research Brief: NVDA ## Executive Summary NVIDIA (NVDA) continues to dominate ...

[Step 4] Node 'critic' [critic_node]
  

In [8]:
print("================================================================================")
print("                         FINAL RELEASED RESEARCH BRIEF                          ")
print("================================================================================")
print(state.output)

                         FINAL RELEASED RESEARCH BRIEF                          
# Research Brief: NVDA  

## Executive Summary  
NVIDIA (NVDA) continues to dominate the AI‑driven compute market, delivering **105 % YoY revenue growth** and ultra‑high profitability (**≈75 % gross margin**, **≈66 % operating margin**). The company generates free‑cash‑flow that is essentially equal to its revenue (~$96 B), underpinning a **$99.3 B share‑repurchase authority** and a modest quarterly dividend.  

Technicals show the stock trading **above both the 50‑day ($222.46) and 200‑day ($202.02) SMAs**, confirming a **bullish uptrend**, while the 14‑day RSI of **49.5** signals a neutral, consolidating momentum. The stock’s beta of **2.22** implies heightened sensitivity to market swings.  

Recent news catalysts—including a **record $150 bn addition to the share‑buyback program**, a **Microsoft‑NVIDIA consumer‑PC partnership**, the launch of an **open‑source AI‑safety platform**, and potential **Apple

In [9]:
from research_agent.memory.session import ResearchSession

# Demonstrate the persistent SQLite conversation memory with follow-up inquiry
thread_id = "master_walkthrough_nvda_session"
session = ResearchSession(thread_id=thread_id)

follow_up = (
    "What is the active company we researched, and how does its gross margin"
    " compare to standard semiconductor industry averages?"
)
print(f"Submitting Follow-up Query to ResearchSession: '{follow_up}'\n")

answer = session.ask(follow_up, show=False)
print("--- SESSION MEMORY RESPONSE ---")
print(answer)

Submitting Follow-up Query to ResearchSession: 'What is the active company we researched, and how does its gross margin compare to standard semiconductor industry averages?'

--- SESSION MEMORY RESPONSE ---
Which company do you mean? Mention a ticker or company name, for example 'What about NVDA's debt?'
